# 🚨 PADS — Proximity-Based Abandoned Object Detection
### GPU-Accelerated Pipeline on NVIDIA T4

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

---
**What this notebook does:**
1. Verifies T4 GPU is available
2. Mounts your Google Drive (where your PADS project lives)
3. Installs all dependencies
4. Builds a GPU-optimised config (YOLOv8x + BotSORT + FP16)
5. Runs the full PADS pipeline on your video
6. Displays the annotated output video and alarm snapshots in-notebook
7. Packages results for download

## ① GPU Verification

In [ ]:
import subprocess, torch

# Print GPU info from nvidia-smi
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Hard check — stops here if no GPU is attached
assert torch.cuda.is_available(), (
    "No GPU found!  Go to Runtime → Change runtime type → T4 GPU"
)

gpu = torch.cuda.get_device_properties(0)
print(f"GPU   : {gpu.name}")
print(f"VRAM  : {gpu.total_memory / 1e9:.1f} GB")
print(f"CUDA  : {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")
print("Ready to run PADS on GPU.")

## ② Mount Google Drive

Your PADS project folder must be uploaded to Google Drive.
Expected structure inside Drive:
```
MyDrive/
  PADS/
    main.py
    modules/
    state/
    utils/
    SampleVideos/
      luggage.mp4
      ...
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

## ③ Set Project Path

Change `PADS_PATH` below if your folder is in a different location inside Drive.

In [ ]:
import os, sys

# ─── CHANGE THIS if your PADS folder is somewhere else in Drive ───
PADS_PATH = "/content/drive/MyDrive/PADS"
# ──────────────────────────────────────────────────────────────────

assert os.path.exists(PADS_PATH), (
    f"PADS folder not found at: {PADS_PATH}\n"
    "Upload your PADS folder to Google Drive first."
)

os.chdir(PADS_PATH)
if PADS_PATH not in sys.path:
    sys.path.insert(0, PADS_PATH)

os.makedirs("output", exist_ok=True)
os.makedirs("logs",   exist_ok=True)

print(f"Working directory : {os.getcwd()}")
print(f"Project files     : {[f for f in os.listdir('.') if not f.startswith('.')]}")

## ④ Install Dependencies

In [ ]:
# Core packages
!pip install ultralytics opencv-python-headless pyyaml numpy -q

# ByteTrack / BotSORT dependency
!pip install lap -q

# Verify
import cv2, yaml
from ultralytics import YOLO
import ultralytics

print(f"OpenCV       : {cv2.__version__}")
print(f"Ultralytics  : {ultralytics.__version__}")
print("All dependencies ready.")

## ⑤ Configure Your Run

Edit the variables in the box below — these are the **only things you need to change** between runs.

| Variable | What it does |
|---|---|
| `VIDEO_SOURCE` | Path to your video file inside the PADS folder |
| `SCENE_PRESET` | Camera/scene type — picks the right scale + thresholds automatically |
| `INIT_FRAMES` | Frames of empty background before action starts |
| `ABANDON_TIMEOUT_S` | Seconds after separation before alarm fires (3 for short test clips, 120 for live) |
| `YOLO_MODEL` | Detection model — see options below |

In [ ]:
# ════════════════════════════════════════════════════════════════
#  USER SETTINGS — edit these between runs
# ════════════════════════════════════════════════════════════════

VIDEO_SOURCE      = "SampleVideos/luggage.mp4"   # path relative to PADS folder

SCENE_PRESET      = "storefront"  # "storefront" | "indoor" | "outdoor_open"

INIT_FRAMES       = 30            # Frames of clean background before subject enters
                                  # Short test clips: 30.  Long live footage: 72+

ABANDON_TIMEOUT_S = 3             # Seconds until alarm fires after separation
                                  # Short test clips: 3.  Live deployment: 120

# ── YOLO model options (auto-downloaded on first use) ────────────
#   Best accuracy (recommended for T4) : "yolov8x.pt"  or  "yolo11x.pt"
#   Speed/accuracy balance             : "yolov8l.pt"  or  "yolo11l.pt"
#   Fastest (weaker GPU / quick test)  : "yolov8n.pt"  or  "yolo11n.pt"
YOLO_MODEL        = "yolov8x.pt"

# ════════════════════════════════════════════════════════════════

import os
assert os.path.exists(VIDEO_SOURCE), (
    f"Video not found: {VIDEO_SOURCE}\n"
    "Upload the video to SampleVideos/ inside your PADS Drive folder."
)
print(f"Video   : {VIDEO_SOURCE}")
print(f"Scene   : {SCENE_PRESET}")
print(f"Model   : {YOLO_MODEL}")
print(f"T_abandon: {ABANDON_TIMEOUT_S}s")
print("Settings OK.")

## ⑥ Build GPU-Optimised Config

This cell writes `config_gpu.yaml` using your settings above.
Key GPU optimisations applied:
- `device: cuda` — YOLO inference runs on T4 VRAM
- `half: true` — FP16 inference on T4 tensor cores (~2× faster vs FP32)
- `tracker: botsort.yaml` — better re-identification than ByteTrack
- `confidence: 0.25` — safe to lower with a large accurate model

In [ ]:
import yaml

# Scene presets — pixel scale + foreground + association values
# tuned for each camera/environment type
PRESETS = {
    "storefront": {
        "pixel_scale_px_per_m"  : 120,
        "var_threshold"         : 55,
        "detect_shadows"        : True,
        "morph_kernel_size"     : 7,
        "dilate_iterations"     : 3,
        "erode_iterations"      : 2,
        "obj_min_area"          : 4000,
        "proximity_radius_m"    : 2.5,
        "separation_threshold_m": 3.5,
        "grace_period_s"        : 2,
        "window_s"              : 1.0,
        "history"               : 800,
        "pending_timeout_s"     : 6,
        # Ghost-object suppression
        "persistence_threshold" : 120,  # higher for busy urban sidewalk
        "freeze_after_init"     : False, # keep False — day/night lighting changes
    },
    "indoor": {
        "pixel_scale_px_per_m"  : 200,
        "var_threshold"         : 16,
        "detect_shadows"        : False,
        "morph_kernel_size"     : 7,
        "dilate_iterations"     : 4,
        "erode_iterations"      : 3,
        "obj_min_area"          : 10000,
        "proximity_radius_m"    : 1.5,
        "separation_threshold_m": 2.0,
        "grace_period_s"        : 2,
        "window_s"              : 0.8,
        "history"               : 500,
        "pending_timeout_s"     : 8,
        "persistence_threshold" : 100,
        "freeze_after_init"     : True,  # stable indoor lighting — safe to freeze
    },
    "outdoor_open": {
        "pixel_scale_px_per_m"  : 100,
        "var_threshold"         : 60,
        "detect_shadows"        : True,
        "morph_kernel_size"     : 9,
        "dilate_iterations"     : 4,
        "erode_iterations"      : 2,
        "obj_min_area"          : 5000,
        "proximity_radius_m"    : 3.0,
        "separation_threshold_m": 4.0,
        "grace_period_s"        : 4,
        "window_s"              : 1.5,
        "history"               : 1000,
        "pending_timeout_s"     : 10,
        "persistence_threshold" : 130,
        "freeze_after_init"     : False,
    },
}

p = PRESETS[SCENE_PRESET]

gpu_cfg = {
    "input": {
        "source"      : VIDEO_SOURCE,
        "init_frames" : INIT_FRAMES,
    },
    "coordinates": {
        "homography_enabled"   : False,
        "pixel_scale_px_per_m" : p["pixel_scale_px_per_m"],
    },
    "foreground": {
        "method"               : "MOG2",
        "history"              : p["history"],
        "var_threshold"        : p["var_threshold"],
        "detect_shadows"       : p["detect_shadows"],
        "shadow_threshold"     : 0.5,
        "morph_kernel_size"    : p["morph_kernel_size"],
        "dilate_iterations"    : p["dilate_iterations"],
        "erode_iterations"     : p["erode_iterations"],
        "obj_min_area"         : p["obj_min_area"],
        "reference_diff_enabled": False,
        # Persistence heatmap — kills transient ghost blobs
        "persistence_enabled"  : True,
        "persistence_threshold": p["persistence_threshold"],
        "persistence_add"      : 25,
        "persistence_decay"    : 8,
        # Freeze background after init — prevents absorbing stationary objects
        "freeze_after_init"    : p["freeze_after_init"],
    },
    "person": {
        "model"     : YOLO_MODEL,
        "confidence": 0.25,
        "iou"       : 0.45,
        "tracker"   : "botsort.yaml",
        "device"    : "cuda",
        "half"      : True,
        "person_ttl_s": 300,
    },
    "association": {
        "proximity_radius_m": p["proximity_radius_m"],
        "pending_timeout_s" : p["pending_timeout_s"],
        "ownership_lookback_s": max(p["pending_timeout_s"], 8),
    },
    "separation": {
        "separation_threshold_m": p["separation_threshold_m"],
        "grace_period_s"        : p["grace_period_s"],
        "window_s"              : p["window_s"],
    },
    "alarm": {
        "abandon_timeout_s" : ABANDON_TIMEOUT_S,
        "retrieve_window_s" : 5,
    },
    "visualizer": {
        "enabled"            : True,
        "show_foreground_mask": False,
        "show_person_boxes"  : True,
        "show_object_boxes"  : True,
        "show_associations"  : True,
        "show_distances"     : True,
        "show_status_overlay": True,
        "output_video"       : "output/output_gpu.mp4",
        "fps_display"        : True,
    },
}

with open("config_gpu.yaml", "w", encoding="utf-8") as f:
    yaml.dump(gpu_cfg, f, default_flow_style=False, sort_keys=False)

print("config_gpu.yaml written")
print(f"  Model              : {YOLO_MODEL}")
print(f"  Device             : cuda  (FP16)")
print(f"  Scene              : {SCENE_PRESET}")
print(f"  Persistence thresh : {p['persistence_threshold']}")
print(f"  Freeze after init  : {p['freeze_after_init']}")
print(f"  Source             : {VIDEO_SOURCE}")

## ⑦ Run PADS Pipeline

The first run downloads the YOLO model weights (~130 MB for yolov8x). Subsequent runs use the cached copy.

In [ ]:
import time
start = time.time()

!python main.py --config config_gpu.yaml --no-display

elapsed = time.time() - start
print(f"\nTotal wall-clock time: {elapsed:.1f}s")

## ⑦b Live Preview — Watch Detection Happen in Real Time

> **Use this cell instead of ⑦ if you want to see frames stream live inside the notebook.**
> 
> Every `DISPLAY_EVERY` frames the annotated output (or M2 debug tile) is rendered inline.
> Set `SHOW_DEBUG = True` to see the side-by-side detection panel.
> Interrupt the kernel at any time to stop early — results are still usable.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
#  ⑦b  LIVE PREVIEW — edit these two lines, then run the cell
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
DISPLAY_EVERY = 2     # render 1 out of every N frames  (1=every frame, 3=faster)
SHOW_DEBUG    = True  # True = side-by-side M2 debug tile | False = PADS output only
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

import cv2, time, yaml, sys, os, io
import numpy as np
from IPython.display import display, clear_output, Image as IPImage

# Reuse the SAME pipeline helpers as main.py so the live preview never drifts
# from the headless run (cell ⑦). All detection/association fixes live in one place.
from main import match_blobs_to_registry, video_clock, resolve_device

from modules.coordinates    import CoordinateMapper
from modules.foreground     import create_foreground_detector
from modules.scene_init     import SceneInitialiser
from modules.person_tracker import PersonTracker
from modules.association    import AssociationEngine
from modules.separation     import SeparationMonitor
from modules.alarm          import AlarmManager
from state.object_registry  import ObjectRegistry, ObjectState, ObjectStatus
from state.person_registry  import PersonRegistry, PersonStatus
from state.event_log        import EventLog
from utils.visualizer       import Visualizer

def _show(img):
    _, buf = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, 88])
    clear_output(wait=True)
    display(IPImage(data=io.BytesIO(buf.tobytes()).getvalue()))

# ── Load config ─────────────────────────────────────────────────────────
with open("config_gpu.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

source = cfg["input"]["source"]
try:
    source = int(source); is_live = True
except (ValueError, TypeError):
    is_live = False

device, half = resolve_device(cfg.get("person", {}).get("device", "auto"))
cfg.setdefault("person", {})["device"] = device
cfg["person"]["half"] = half
print(f"Device : {device}" + (" [FP16]" if half else ""))

cap = cv2.VideoCapture(source)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {source}")

fps_src = cap.get(cv2.CAP_PROP_FPS) or 24
width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Source : {source}  |  {width}×{height}  @  {fps_src:.1f} fps  |  {total} frames")

# ── Initialise components ─────────────────────────────────────────────────
obj_reg    = ObjectRegistry()
person_reg = PersonRegistry()
event_log  = EventLog("logs/events_live.jsonl")
mapper     = CoordinateMapper(cfg)
fg_det     = create_foreground_detector(cfg)
visualizer = Visualizer(cfg)

scene_init = SceneInitialiser(fg_det, mapper, obj_reg, event_log, cfg)
assoc      = AssociationEngine(mapper, obj_reg, person_reg, event_log, cfg)
sep_mon    = SeparationMonitor(mapper, obj_reg, person_reg, event_log, cfg)
alarm_mgr  = AlarmManager(mapper, obj_reg, person_reg, event_log, cfg)
tracker    = PersonTracker(cfg, mapper, person_reg)

# Seed first_seen on the same (video) timeline used by the loop below.
init_t0 = (cfg.get("input", {}).get("init_frames", 30) / fps_src) if not is_live else time.time()
scene_init.run(cap, now=init_t0)
fg_det.freeze()

min_area         = cfg.get("foreground", {}).get("obj_min_area", 800)
detection_method = cfg.get("foreground", {}).get("method", "MOG2").upper()

# ── Main loop ─────────────────────────────────────────────────────────────
frame_count    = 0
frame_time     = init_t0
last_frame     = None
blob_centroids = []
t_start        = time.time()

print(f"Streaming frames (DISPLAY_EVERY={DISPLAY_EVERY}, SHOW_DEBUG={SHOW_DEBUG})")
print("Interrupt the kernel (■ button) to stop early.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    last_frame  = frame
    frame_count += 1
    frame_time  = video_clock(cap, frame_count, fps_src, is_live)   # video time, not wall-clock

    # M3 — person tracking
    tracker.process(frame, frame_time)
    person_boxes = [p.bbox for p in person_reg.in_frame()]
    fg_det.set_person_regions(person_boxes, frame.shape)

    # M2 — foreground detection
    blobs, fg_mask = fg_det.process(frame)

    new_obj_ids, blob_centroids = match_blobs_to_registry(
        blobs, obj_reg, mapper, min_area, person_boxes, frame_time
    )

    if new_obj_ids:
        assoc.process_new_objects(new_obj_ids, frame_time)
    assoc.retry_pending(frame_time)
    sep_mon.process(frame_time)

    new_alarms = alarm_mgr.tick(frame_time, frame, blob_centroids)
    for alarm in new_alarms:
        print(f"🚨 ALARM {alarm['alarm_id']}  |  Object {alarm['object_id']}")

    if frame_count % DISPLAY_EVERY == 0:
        annotated = visualizer.draw(frame, obj_reg, person_reg, alarm_mgr, frame_time)
        if SHOW_DEBUG:
            tile = visualizer.draw_debug_tile(annotated, fg_mask, blobs, min_area, detection_method)
            _show(tile)
        else:
            _show(annotated)

cap.release()

# ── End-of-video flush ────────────────────────────────────────────────
for person in person_reg.in_frame():
    if person.status in (PersonStatus.OF_INTEREST, PersonStatus.ACTIVE):
        person.in_frame  = False
        person.exit_time = frame_time
        if person.status == PersonStatus.ACTIVE:
            person.status = PersonStatus.EXITED
        person_reg.update(person)

for extra_t in range(1, int(alarm_mgr.T_abandon) + 2):
    flush_alarms = alarm_mgr.tick(frame_time + extra_t, last_frame, blob_centroids)
    for alarm in flush_alarms:
        print(f"🚨 ALARM (flush) {alarm['alarm_id']}  |  Object {alarm['object_id']}")
    if flush_alarms:
        break

elapsed = time.time() - t_start
print(f"\nDone — {frame_count} frames in {elapsed:.1f}s  ({frame_count/elapsed:.1f} fps effective)")
print(f"Alarms fired : {len(alarm_mgr.all_alarms())}")


## ⑧ View Output Video

In [ ]:
import os
from IPython.display import HTML, display
from base64 import b64encode

RAW = "output/output_gpu.mp4"
H264 = "output/output_gpu_h264.mp4"

if not os.path.exists(RAW):
    print("Output video not found. Check pipeline output above for errors.")
else:
    # Re-encode to H.264 so browsers can play it inline
    os.system(f'ffmpeg -y -i "{RAW}" -vcodec libx264 -crf 20 -preset fast "{H264}" -loglevel error')

    size_mb = os.path.getsize(H264) / 1e6
    print(f"Output video: {H264}  ({size_mb:.1f} MB)")

    if size_mb < 50:   # Embed directly for small files
        data = open(H264, 'rb').read()
        url  = 'data:video/mp4;base64,' + b64encode(data).decode()
        display(HTML(f'''
            <h3>PADS — Annotated Output</h3>
            <video width="960" controls style="border:2px solid #333; border-radius:6px">
              <source src="{url}" type="video/mp4">
            </video>
        '''))
    else:
        print("File too large to embed. Download it using the cell below.")

## ⑨ View Alarm Snapshots

In [ ]:
import glob, json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

snapshots  = sorted(glob.glob("logs/ALARM_*_snapshot.jpg"))
alarm_meta = sorted(glob.glob("logs/ALARM_*_alarm.json"))

if not snapshots:
    print("No alarms were triggered.")
    print("Try reducing abandon_timeout_s or checking the pipeline log above.")
else:
    n = len(snapshots)
    print(f"{n} alarm(s) triggered")

    fig, axes = plt.subplots(1, n, figsize=(9 * n, 6))
    if n == 1:
        axes = [axes]

    for ax, snap, jpath in zip(axes, snapshots, alarm_meta):
        img = mpimg.imread(snap)
        ax.imshow(img)

        meta = {}
        if os.path.exists(jpath):
            with open(jpath) as f:
                meta = json.load(f)

        label = (
            f"ALARM {meta.get('alarm_id','?')}\n"
            f"Object {meta.get('object_id','?')}  "
            f"Person {meta.get('person_id','?')}"
        )
        ax.set_title(label, fontsize=11, color="red", fontweight="bold")
        ax.axis("off")

    plt.suptitle("PADS — Abandoned Object Alarms", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## ⑩ Event Log Summary

In [ ]:
import json

LOG = "logs/events.jsonl"

if not os.path.exists(LOG):
    print("No event log found.")
else:
    events = []
    with open(LOG) as f:
        for line in f:
            line = line.strip()
            if line:
                events.append(json.loads(line))

    print(f"Total events logged: {len(events)}\n")

    # Count by type
    from collections import Counter
    counts = Counter(e.get("event_type", "UNKNOWN") for e in events)
    for etype, cnt in counts.most_common():
        print(f"  {etype:<30} {cnt}")

    # Print alarm details
    alarms = [e for e in events if "ALARM" in e.get("event_type", "")]
    if alarms:
        print(f"\nAlarm details:")
        for a in alarms:
            print(f"  {a}")

## ⑪ Benchmark — GPU vs CPU Speed

Run this cell **after** a successful GPU run to measure how much faster CUDA is vs CPU for YOLO inference.

In [ ]:
import time, cv2, torch
from ultralytics import YOLO

MODEL   = YOLO_MODEL
N_BENCH = 20   # frames to benchmark

cap = cv2.VideoCapture(VIDEO_SOURCE)
frames = []
for _ in range(N_BENCH):
    ret, f = cap.read()
    if ret: frames.append(f)
cap.release()

if not frames:
    print("Could not read video frames for benchmark.")
else:
    model = YOLO(MODEL)

    # Warm up GPU
    model.predict(frames[0], device="cuda", half=True, verbose=False)

    # GPU FP16
    t0 = time.time()
    for f in frames:
        model.predict(f, device="cuda", half=True, verbose=False)
    gpu_ms = (time.time() - t0) / len(frames) * 1000

    # GPU FP32
    t0 = time.time()
    for f in frames:
        model.predict(f, device="cuda", half=False, verbose=False)
    gpu32_ms = (time.time() - t0) / len(frames) * 1000

    # CPU
    t0 = time.time()
    for f in frames:
        model.predict(f, device="cpu", verbose=False)
    cpu_ms = (time.time() - t0) / len(frames) * 1000

    print(f"Model : {MODEL}")
    print(f"Frames: {len(frames)}")
    print()
    print(f"CPU   FP32  : {cpu_ms:6.1f} ms/frame   ({1000/cpu_ms:5.1f} FPS)")
    print(f"T4    FP32  : {gpu32_ms:6.1f} ms/frame   ({1000/gpu32_ms:5.1f} FPS)")
    print(f"T4    FP16  : {gpu_ms:6.1f} ms/frame   ({1000/gpu_ms:5.1f} FPS)  <-- used by PADS")
    print()
    print(f"Speedup GPU FP16 vs CPU: {cpu_ms/gpu_ms:.1f}x faster")

## ⑫ Download Results

In [ ]:
import zipfile, glob
from google.colab import files

ZIP = "pads_results.zip"

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for pattern in [
        "output/output_gpu_h264.mp4",
        "output/output_gpu.mp4",
        "logs/ALARM_*_snapshot.jpg",
        "logs/ALARM_*_alarm.json",
        "logs/events.jsonl",
        "config_gpu.yaml",
    ]:
        for path in glob.glob(pattern):
            if os.path.exists(path):
                zf.write(path)
                print(f"  + {path}")

print(f"\nDownloading {ZIP} ...")
files.download(ZIP)

---
## Quick Reference

### Switching videos
Edit `VIDEO_SOURCE` in cell ⑤, then re-run cells ⑤ → ⑦.

### Switching scenes
Change `SCENE_PRESET` in cell ⑤:
- `"storefront"` — outdoor urban sidewalk, elevated wide-angle camera
- `"indoor"` — indoor space (restaurant, airport hall), overhead camera
- `"outdoor_open"` — open outdoor area (park, plaza), distant camera

### Model options
| Model | Accuracy | Speed on T4 FP16 | Notes |
|---|---|---|---|
| `yolov8n.pt` | Lowest | ~120 FPS | Only for very fast testing |
| `yolov8s.pt` | OK | ~90 FPS | Decent for quick runs |
| `yolov8l.pt` | Good | ~55 FPS | Good balance |
| **`yolov8x.pt`** | **Best v8** | **~35 FPS** | **Recommended** |
| `yolo11l.pt` | Better | ~50 FPS | Newer architecture |
| `yolo11x.pt` | Best overall | ~30 FPS | Maximum accuracy |

### Live deployment timeouts
For real CCTV footage change these in cell ⑤:
```python
INIT_FRAMES       = 72    # 3s warmup
ABANDON_TIMEOUT_S = 120   # 2 minute alarm threshold
```